In [ ]:
import torch
import numpy as np

import scienceplots
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm
from scipy.stats import multivariate_normal

from models import MLP
from utils import GaussianMixture, gmm_with_pdf_and_scores, gmm_score_at_t

plt.style.use(["science", "no-latex"])

seed = 42
np.random.seed(seed)

Assume that our data is $x_{0} \sim p_{0}, x_{0} \in \mathbb{R}^{d}$. As mentioned, we diffuse using the OU SDE

\begin{align*}
    dx = -xdt + \sqrt{2}dw
\end{align*}

and we have following marginal

\begin{align*}
    x_{t} \mid x_{0} \sim \mathcal{N}\left(e^{-t}x_{0}, (1 - e^{-2t})I\right)
\end{align*}

In this notebook, we will assume that $p_{0}$ is mixture of 2 gaussian's whose density is given by
\begin{align*}
    p_{0}(x_{0}) = \alpha_{1}\mathcal{N}(\mu_{1},\Sigma_{1}) + \alpha_{2}\mathcal{N}(\mu_{2},\Sigma_{2})
\end{align*}

Because of this, marginal at any time $t$ will be

\begin{align*}
    p_{t}(x_{t}) &= \int_{x_{0}}p_{t|0}(x_{t}|x_{0})p_{0}(x_{0})dx_{0} \\
    &= \alpha_{1}\int_{x_0}\mathcal{N}(x_{0}; \mu_{1},\Sigma_{1})\mathcal{N}\left(x_{t}; e^{-t}x_{0}, (1 - e^{-2t})I\right)dx_{0} + \alpha_{2}\int_{x_{0}}\mathcal{N}(x_{0};\mu_{2},\Sigma_{2})\mathcal{N}\left(x_{t}; e^{-t}x_{0}, (1 - e^{-2t})I\right)dx_{0} \\
    &= \alpha_{1}\mathcal{N}\left(e^{-t}\mu_{1}, e^{-2t}\Sigma_{1} + (1 - e^{-2t})I\right) + \alpha_{2}\mathcal{N}\left(e^{-t}\mu_{2}, e^{-2t}\Sigma_{2} + (1 - e^{-2t})I\right)
\end{align*}

So the marginal is also mixture of gaussian. Using this information, let's visualize the marginal at different time steps. For the experiments in this notebook, we will assume that $\alpha_{1} = \alpha_{2} = 0.5, \mu_{1} = (-2, -2), \Sigma_{1} = 0.5I, \mu_{2} = (2, 2), \Sigma_{2} = 0.5I$. In future, we might change the gaussian weights.

Now let's visualize how diffusion happens as time passes.

In [ ]:
# define the initial parameters of the gaussian mixture model
initial_alphas = np.array([0.5, 0.5])
initial_means = np.array([[-2, -2], [2, 2]])
initial_covs = np.array([
    [[0.5, 0],[0, 0.5]],
    [[0.5, 0], [0, 0.5]]
])

# don't know how much I will use but let's see
initial_gmm = GaussianMixture(initial_alphas, initial_means, initial_covs)


In [ ]:
# draw the initial samples
initial_samples = initial_gmm.sample(1000)

# we will draw the samples and contours from t = 0 to t = 10. We will plot this in 20 time steps
# timesteps = np.linspace(0, 10, 20)
timesteps = np.linspace(0, 2, 15)
timesteps = np.append(timesteps, np.linspace(3, 10, 5))

# we should do this subplot with 2 rows and 10 columns
# another figure size so that the plots are not too small
fig, axes = plt.subplots(2, 10, figsize=(24, 7))    

for idx, t in enumerate(timesteps):
    row_idx = idx // 10
    col_idx = idx % 10

    X, Y, Z, _, _, _ = gmm_with_pdf_and_scores(initial_alphas, initial_means, initial_covs, t)

    # for samples, we will do the noise addition to the initial samples
    samples = np.exp(-t) * initial_samples + np.sqrt(1 - np.exp(-2 * t)) * np.random.normal(size=initial_samples.shape)

    # plot the samples and contours
    axes[row_idx, col_idx].contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
    axes[row_idx, col_idx].set_title(f"t = {t:.2f}")
    axes[row_idx, col_idx].set_xticks([])
    axes[row_idx, col_idx].set_yticks([])

fig.suptitle("Contours of density at Different Time Steps", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# draw the initial samples
initial_samples = initial_gmm.sample(1000)

# we will draw the samples and contours from t = 0 to t = 10. We will plot this in 20 time steps
# timesteps = np.linspace(0, 10, 20)
timesteps = np.linspace(0, 2, 15)
timesteps = np.append(timesteps, np.linspace(3, 10, 5))

# we should do this subplot with 2 rows and 10 columns
# another figure size so that the plots are not too small
fig, axes = plt.subplots(2, 10, figsize=(24, 7))    

for idx, t in enumerate(timesteps):
    row_idx = idx // 10
    col_idx = idx % 10

    X, Y, Z, _, _, _ = gmm_with_pdf_and_scores(initial_alphas, initial_means, initial_covs, t)

    # for samples, we will do the noise addition to the initial samples
    samples = np.exp(-t) * initial_samples + np.sqrt(1 - np.exp(-2 * t)) * np.random.normal(size=initial_samples.shape)

    # plot the samples and contours
    axes[row_idx, col_idx].scatter(samples[:, 0], samples[:, 1], alpha=0.5, label='Samples', s=2)
    axes[row_idx, col_idx].contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
    axes[row_idx, col_idx].set_title(f"t = {t:.2f}")
    axes[row_idx, col_idx].set_xticks([])
    axes[row_idx, col_idx].set_yticks([])

fig.suptitle("Contours and samples of density at Different Time Steps", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# we will draw the samples and contours from t = 0 to t = 10. We will plot for 9 time steps
timesteps = [0.00, 0.14, 0.29, 0.43, 0.57, 1.00, 1.14, 6.75, 10.00]

# we should do this subplot with 3 rows and 3 columns
fig, axes = plt.subplots(3, 3, figsize=(15, 15))    

for idx, t in enumerate(timesteps):
    row_idx = idx // 3
    col_idx = idx % 3

    X, Y, Z, U, V, _ = gmm_with_pdf_and_scores(initial_alphas, initial_means, initial_covs, t)

    # plot the samples and contours
    axes[row_idx, col_idx].contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
    axes[row_idx, col_idx].quiver(X, Y, U, V, color='red', alpha=0.5)
    axes[row_idx, col_idx].set_title(f"t = {t:.2f}")
    axes[row_idx, col_idx].set_xticks([])
    axes[row_idx, col_idx].set_yticks([])

fig.suptitle("Contours and Original Score Vectors at Different Time Steps", fontsize=16)
plt.tight_layout()
plt.show()

## TRAINING

We will do score estimation using 5000 samples for 50 timesteps between 0 to 10. Solving the following objective

\begin{align*}
    \arg\min_{\theta} \mathbb{E}_{t \sim [0, T]}\lambda(t)\mathbb{E}_{x_{0} \sim p_{0}}\mathbb{E}_{x_{t} \sim p(x_{t} \mid x_{0})}\left[\lVert s_{\theta}(x_{t}, t) - \nabla_{x} \log p(x_{t}|x_{0})\Vert^{2}\right]
\end{align*}

As $x_{t} \mid x_{0}$ is gaussian, we have

\begin{align*}
    \arg\min_{\theta} \mathbb{E}_{t \sim [0, T]}\lambda(t)\mathbb{E}_{x_{0} \sim p_{0}}\mathbb{E}_{x_{t} \sim p(x_{t} \mid x_{0})}\left[\Vert s_{\theta}(x_{t}, t) + \frac{x_{t} - e^{-t}x_{0}}{1 - e^{-2t}}\Vert^{2}\right]
\end{align*}

Writing $x_{t} = e^{-t}x_{0} + \sqrt{1 - e^{-2t}}\epsilon$ where $\epsilon \sim \mathcal{N}(0, I)$, then we have

\begin{align*}
    \arg\min_{\theta} \mathbb{E}_{t \sim [0, T]}\lambda(t)\mathbb{E}_{x_{0} \sim p_{0}}\mathbb{E}_{\epsilon \sim \mathcal{N}(0, I)}\left[\Vert s_{\theta}(x_{t}, t) + \frac{\epsilon}{\sqrt{1 - e^{-2t}}}\Vert^{2}\right]
\end{align*}

Usually

\begin{align*}
    \lambda(t) \propto \frac{1}{\mathbb{E}[\Vert \nabla_{x} \log p(x_{t}|x_{0}) \Vert^{2}]}
\end{align*}

In our case, $\nabla_{x} \log p(x_{t}|x_{0}) = \frac{\epsilon}{\sqrt{1 - e^{-2t}}}$ implies we have


\begin{align*}
    \lambda(t) \propto \frac{(1 - e^{-2t})}{\mathbb{E}[\Vert \epsilon \Vert^{2}]} = \frac{(1 - e^{-2t})}{d}
\end{align*}

In [ ]:
device = torch.device("cpu")

model = MLP(input_dim=2, hidden_dim=128, output_dim=2)
model.load_state_dict(torch.load("gmm_score_estimation.bin"))
model.to(device)

In [ ]:
# let's compare predicted score and original score
tt = [10, 4, 2, 1, 0.7, 0.1, 0.05]

# set to eval mode
model.eval()

for t in tt:
    # original
    X, Y, Z, orig_score_x, orig_score_y, inp_points = gmm_with_pdf_and_scores(initial_alphas, initial_means, initial_covs, t)

    # predicted
    inp_points_tensor = torch.tensor(inp_points, dtype=torch.float32, device=device)
    with torch.no_grad():
        t_tensor = torch.tensor([t] * len(inp_points_tensor), dtype=torch.float32, device=device)
        pred_scores = model(inp_points_tensor, t_tensor).cpu().numpy()
        pred_scores_x = pred_scores[:, 0].reshape(X.shape)
        pred_scores_y = pred_scores[:, 1].reshape(Y.shape)

    # calculate the MSE
    original_scores = np.hstack((orig_score_x.ravel()[:, None], orig_score_y.ravel()[:, None]))
    mse = np.mean((original_scores - pred_scores) ** 2)

    # plot and compare
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
    plt.quiver(X, Y, orig_score_x, orig_score_y, color='red', alpha=0.5)
    plt.title(f"Original Score at t = {t:.2f}")
    plt.xticks([])
    plt.yticks([])

    plt.subplot(1, 2, 2)
    plt.contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
    plt.quiver(X, Y, pred_scores_x, pred_scores_y, color='blue', alpha=0.5)
    plt.title(f"Predicted Score at t = {t:.2f}")
    plt.xticks([])
    plt.yticks([])

    plt.suptitle(f"MSE = {round(mse, 2)}")
    plt.tight_layout()
    plt.show()

## REVERSE SDE AND SAMPLING

The reverse SDE for OU process that we are considering is

\begin{align*}
    dx = (-x - 2\nabla_{x}\log p_{t}(x))dt + \sqrt{2}d\bar{w}
\end{align*}

where $\bar{w}$ is a standard Weiner process when time flown backwards. Suppose we are running the forward SDE till $T$ that is $[0, T]$ and we discretize this time in $N$ steps. So specifically we are running at times

\begin{align*}
    0, \frac{T}{N}, \frac{2T}{N}, \dots, T
\end{align*}

that is $k\frac{T}{N}$ where $0 \leq k \leq N$. Defining $\Delta t = \frac{T}{N}$, then discretizing the reverse SDE, we get

\begin{align*}
    x_{(k - 1)\Delta t} - x_{k\Delta t} = \left(- x_{k\Delta t} - 2\nabla_{x}\log p_{k\Delta t}\left(x_{k\Delta t}\right)\right)\left((k-1)\Delta t - k\Delta t\right) + \sqrt{2}\left(W\left((k - 1)\Delta t\right) - W\left(k\Delta t\right)\right)
\end{align*}

where $k$ runs from $N$ to $1$. Now $W\left((k - 1)\Delta t\right) - W\left(k\Delta t\right) = \sqrt{\Delta t}\epsilon$ where $\epsilon \sim \mathcal{N}(0, I)$. So we have

\begin{align*}
    x_{(k - 1)\Delta t} - x_{k\Delta t} = \left(x_{k\Delta t} + 2\nabla_{x} \log p_{k\Delta t}(x_{k\Delta t})\right)\Delta t + \sqrt{2\Delta t}\epsilon
\end{align*}

So we start with $x_{T}$ ($k = N$), using score at time $T$ evaluated at $x_{T}$ we go to $x_{(N - 1)\Delta t}$. Similarly, at the end we have $x_{\Delta t}$, using the score at time $\Delta t$ evaluated at $x_{\Delta t}$ we go to $x_{0}$ generating a single sample from $p_{0}$.


We have $T = 10$ and $N=100$. This is the same configuration we are using for model training.

In [ ]:
total_time = 10
timesteps = 100

# set to eval mode
model.eval()

del_t = total_time / timesteps

# sample the point from N(0, 1) - the converging distribution
x = np.random.normal(size=(1, 2)).astype(np.float32)

points = [x.copy()]

# we will plot the direction of score, the update direction and contour at time steps
# plot_times = [10, 9.9, 6.0, 3.0, 1.0, 0.7, 0.5, 0.4, 0.1]
plot_steps = [100, 99, 60, 30, 10, 7, 5, 4, 1]
cidx = 0
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for k in range(timesteps, 0, -1):
    t = k * del_t

    X, Y, Z, _, _, _ = gmm_with_pdf_and_scores(initial_alphas, initial_means, initial_covs, t)

    # use x to predict the score
    x_tensor = torch.tensor(x, dtype=torch.float32, device=device)
    with torch.no_grad():
        t_tensor = torch.tensor([t], dtype=torch.float32, device=device)
        score = model(x_tensor, t_tensor).cpu().numpy()

    # use original score
    # score = gmm_score_at_t(initial_alphas, initial_means, initial_covs, t, x)

    update = (x + 2 * score) * del_t + np.sqrt(2 * del_t) * np.random.normal(0, 1)

    if k in plot_steps:
        row_idx = cidx // 3
        col_idx = cidx % 3

        axes[row_idx, col_idx].contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
        axes[row_idx, col_idx].quiver(x[0, 0], x[0, 1], score[0, 0], score[0, 1], color='red', alpha=0.5, label='Score Direction')
        axes[row_idx, col_idx].quiver(x[0, 0], x[0, 1], update[0, 0], update[0, 1], color='blue', alpha=0.5, label='Update Direction')
        axes[row_idx, col_idx].set_title(f"t = {t:.2f}")
        axes[row_idx, col_idx].legend()

        cidx += 1

    x = x + update
    points.append(x.copy())

fig.suptitle("Contours, Score and Update Directions at Different Time Steps", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# using the reverse SDE, we will now sample 1000 points
n_samples = 1000

# set to eval mode
model.eval()

xp = np.random.normal(size=(n_samples, 2)).astype(np.float32)

predictor_points = [xp.copy()]

for k in range(timesteps, 0, -1):
    t = k * del_t

    # use x to predict the score
    x_tensor = torch.tensor(xp, dtype=torch.float32, device=device)
    with torch.no_grad():
        t_tensor = torch.tensor([t] * n_samples, dtype=torch.float32, device=device)
        score = model(x_tensor, t_tensor).cpu().numpy()

    update = (xp + 2 * score) * del_t + np.sqrt(2 * del_t) * np.random.normal(size=xp.shape)
    xp = xp + update

    predictor_points.append(xp.copy())

print(xp.shape)

predictor_points = np.array(predictor_points)
print(predictor_points.shape)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

plot_steps = [100, 30, 10, 7, 5, 4, 3, 2, 1]
cidx = 0

for k in plot_steps:

    # get row and col
    row_idx = cidx // 3
    col_idx = cidx % 3

    # time
    t = k * del_t

    X, Y, Z, _, _, _ = gmm_with_pdf_and_scores(initial_alphas, initial_means, initial_covs, t)

    axes[row_idx, col_idx].contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
    axes[row_idx, col_idx].scatter(predictor_points[timesteps - k][:, 0], predictor_points[timesteps - k][:, 1], alpha=0.5, label='Samples', s=2)
    axes[row_idx, col_idx].set_title(f"t = {t:.2f}")
    axes[row_idx, col_idx].set_xticks([])
    axes[row_idx, col_idx].set_yticks([])

    cidx += 1

fig.suptitle("Contours and samples of density at Different Time Steps", fontsize=16)
plt.tight_layout()
plt.show()

## PREDICTOR-CORRECTOR (PC) SAMPLING

In this, after each discretized reverse SDE step, we run a $M$ steps of langevin dynamics. This was also proposed in the paper, so let's see how it works.

In [ ]:
# using the reverse SDE, we will now sample 1000 points
n_samples = 1000

# langevin dynamics parags
langevin_steps = 20
langevin_step_size = 0.2

# set to eval mode
model.eval()

xpc = np.random.normal(size=(n_samples, 2)).astype(np.float32)

for k in range(timesteps, 0, -1):
    t = k * del_t

    # predictor phase
    # use x to predict the score
    x_tensor = torch.tensor(xpc, dtype=torch.float32, device=device)
    with torch.no_grad():
        t_tensor = torch.tensor([t] * n_samples, dtype=torch.float32, device=device)
        score = model(x_tensor, t_tensor).cpu().numpy()

    update = (xpc + 2 * score) * del_t + np.sqrt(2 * del_t) * np.random.normal(size=xpc.shape)
    xpc = xpc + update

    # starting at this, we will do langevin dynamics. The time we consider is (k - 1) * del_t.
    # this is corrector phase.
    for _ in range(langevin_steps):
        x_tensor = torch.tensor(xpc, dtype=torch.float32, device=device)
        with torch.no_grad():
            t_tensor = torch.tensor([(k - 1) * del_t] * n_samples, dtype=torch.float32, device=device)
            score = model(x_tensor, t_tensor).cpu().numpy()

        xpc = xpc + langevin_step_size * score + np.sqrt(2 * langevin_step_size) * np.random.normal(size=xpc.shape)

print(xpc.shape)

In [ ]:
X, Y, Z, _, _, _ = gmm_with_pdf_and_scores(initial_alphas, initial_means, initial_covs, 0)

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

axes[0].contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
axes[0].scatter(initial_samples[:, 0], initial_samples[:, 1], alpha=0.5, label="Original Samples", s=2)
axes[0].set_title("Original Samples at t=0")

axes[1].contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
axes[1].scatter(xp[:, 0], xp[:, 1], alpha=0.5, label="Original Samples", s=2)
axes[1].set_title("Generated Samples (P) at t=0")

axes[2].contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
axes[2].scatter(xpc[:, 0], xpc[:, 1], alpha=0.5, label="Original Samples", s=2)
axes[2].set_title("Generated Samples (PC) at t=0")

plt.tight_layout()
plt.show()